<a href="https://colab.research.google.com/github/viictorsauraa/AprendizajeporRefuerzo_SauraCarmonaCortesGrupo7/blob/main/k_brazos/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problema del Bandido de k-brazos

## Descripción

Este notebook es el punto de entrada para el estudio del **problema del bandido de k-brazos** (*Multi-Armed Bandit*, MAB), un problema clásico de toma de decisiones secuenciales bajo incertidumbre en el contexto del Aprendizaje por Refuerzo.

El problema modela una máquina tragaperras con $k$ brazos (acciones), donde cada brazo $i$ tiene una distribución de recompensa desconocida $\mathcal{P}_i$. El agente selecciona un brazo en cada paso de tiempo y recibe una recompensa $r_t \sim \mathcal{P}_{a_t}$. El objetivo es maximizar la recompensa acumulada $\sum_{t=1}^T r_t$ encontrando el equilibrio óptimo entre **exploración** (aprender sobre brazos desconocidos) y **explotación** (elegir el brazo con mayor recompensa estimada).

El rendimiento se evalúa mediante el **rechazo acumulado** (*regret*):

$$R(T) = T\mu^* - \mathbb{E}\left[\sum_{t=1}^{T} \mu_{a_t}\right]$$

donde $\mu^* = \max_i \mu_i$ y $\mu_i = \mathbb{E}[\mathcal{P}_i]$. Lai & Robbins (1985) demostraron que todo algoritmo consistente satisface el límite inferior:

$$\liminf_{T \to \infty} \frac{R(T)}{\ln T} \geq C = \sum_{i:\, \mu_i < \mu^*} \frac{\mu^* - \mu_i}{\text{KL}(\mathcal{P}_i \| \mathcal{P}^*)}

## Preparación del entorno

In [ ]:
#@title Copiar el repositorio.
import os

if 'google.colab' in str(get_ipython()):
    if not os.path.exists('/content/AprendizajeporRefuerzo_SauraCarmonaCortesGrupo7'):
        !git clone https://github.com/viictorsauraa/AprendizajeporRefuerzo_SauraCarmonaCortesGrupo7.git
    %cd /content/AprendizajeporRefuerzo_SauraCarmonaCortesGrupo7/k_brazos


In [ ]:
%%capture
#@title Instalar dependencias
!pip install -q numpy matplotlib seaborn tqdm

In [ ]:
#@title Verificar instalación
import sys
sys.path.insert(0, 'src')

import numpy as np

from algorithms import Algorithm, EpsilonGreedy, EpsilonGreedyinitialization, EpsilonDecaimiento, UCB1, Softmax
from arms import ArmNormal, ArmBinomial, ArmBernoulli, Bandit
from plotting import plot_average_rewards, plot_optimal_selections, plot_regret, plot_arm_statistics

print("Entorno listo. Todas las clases importadas correctamente.")

## Estructura del estudio

El estudio se organiza en los siguientes notebooks. Para cada tipo de distribución de brazos se realizan dos tipos de análisis: (1) el estudio de la familia ε-Greedy y (2) una comparativa completa de todos los algoritmos implementados.

---

### Familia ε-Greedy

#### [bandit_experiment.ipynb](https://colab.research.google.com/github/viictorsauraa/AprendizajeporRefuerzo_SauraCarmonaCortesGrupo7/blob/main/k_brazos/bandit_experiment.ipynb)
Estudio de la familia ε-Greedy (ε = 0, 0.01, 0.1) **sin inicialización** sobre brazos de distribución Normal. Se analiza el efecto del parámetro ε y se identifica la gráfica más relevante para evaluar el rendimiento.

#### [bandit_experiment_Greedy_con_inicializacion.ipynb](https://colab.research.google.com/github/viictorsauraa/AprendizajeporRefuerzo_SauraCarmonaCortesGrupo7/blob/main/k_brazos/bandit_experiment_Greedy_con_inicializacion.ipynb)
Mismo estudio que el anterior pero **con inicialización round-robin** (selección de cada brazo al menos una vez antes de aplicar la política). Permite observar cómo la exploración inicial mejora el rendimiento, especialmente para ε = 0.

---

### Estudio comparativo por tipo de distribución

#### [bandit_experiment_Normal.ipynb](https://colab.research.google.com/github/viictorsauraa/AprendizajeporRefuerzo_SauraCarmonaCortesGrupo7/blob/main/k_brazos/bandit_experiment_Normal.ipynb)
Comparativa completa de ε-Greedy, ε-Decaimiento, UCB1 y Softmax con brazos de distribución **Normal** $\mathcal{N}(\mu, \sigma^2)$. Incluye variaciones del número de brazos y de la varianza para analizar la robustez de cada algoritmo.

#### [bandit_experiment_Binomial.ipynb](https://colab.research.google.com/github/viictorsauraa/AprendizajeporRefuerzo_SauraCarmonaCortesGrupo7/blob/main/k_brazos/bandit_experiment_Binomial.ipynb)
Comparativa completa de algoritmos con brazos de distribución **Binomial** $\mathcal{B}(n, p)$, donde cada brazo representa el número total de éxitos en $n$ ensayos con probabilidad $p$.

#### [bandit_experiment_Bernoulli.ipynb](https://colab.research.google.com/github/viictorsauraa/AprendizajeporRefuerzo_SauraCarmonaCortesGrupo7/blob/main/k_brazos/bandit_experiment_Bernoulli.ipynb)
Comparativa completa de algoritmos con brazos de distribución **Bernoulli** (caso particular de la Binomial con $n=1$). Modela situaciones binarias como el CTR (*Click-Through Rate*) en publicidad online.

---

## Algoritmos implementados

| Algoritmo | Clase | Parámetros principales | Inicialización |
|---|---|---|---|
| ε-Greedy | `EpsilonGreedy` | `epsilon` ∈ [0, 1] | No |
| ε-Greedy + init | `EpsilonGreedyinitialization` | `epsilon` ∈ [0, 1] | Round-robin |
| ε-Decaimiento | `EpsilonDecaimiento` | `epsilon_initial`, `lambda_`, `epsilon_min` | Round-robin |
| UCB1 | `UCB1` | `c` >= 0 | Round-robin |
| Softmax (Gibbs) | `Softmax` | `tau` > 0 | No |

## Tipos de brazos implementados

| Distribución | Clase | Valor esperado | Varianza |
|---|---|---|---|
| Normal | `ArmNormal` | $\mu$ | $\sigma^2$ |
| Binomial | `ArmBinomial` | $np$ | $np(1-p)$ |
| Bernoulli | `ArmBernoulli` | $p$ | $p(1-p)$ |
